# 🦾 Lab 3: Stand on the Shoulders of Giants — Transfer Learning

In Lab 2 you built a CNN from scratch and fought overfitting with augmentation. You gained a few points. Respectable. 🥊

Now the industry secret: **don't start from scratch.** We take **ResNet18** — pretrained on 1.2 million ImageNet photos — and bend it to our cats-vs-dogs task. Same data as Lab 2. Watch what happens.

## 📚 The plan (2 acts + a coronation)
1. **🔮 Act 1 — The Oracle:** the pretrained model guesses what's in YOUR photo, out of 1,000 categories. Zero training.
2. **🦾 Act 2 — The Transplant:** freeze the giant's body, replace its final layer, train only that.
3. **👑 The Scoreboard:** scratch vs augmented vs transfer — three models, same data, one champion.

⚡ **Recommended:** GPU runtime.

📦 **Dataset:** CIFAR-10 (automatic) + optionally your own photo for Act 1.

In [ ]:
!pip install torch torchvision matplotlib numpy tqdm pillow scikit-learn

### 🛠️ Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torchvision import models
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", "✅ GPU" if device.type == 'cuda' else "❌ CPU (works, just slower)")

### 🔮 Act 1: The Oracle — 1,000-Class Vision, Zero Training

ResNet18 already knows 1,000 ImageNet categories: dog breeds, foods, cars, instruments... Let's load it **fully pretrained** and ask it about a photo.

Upload any photo as `my_photo.jpg` (your cat, your lunch, your car 🚗) — or the backup image is used.

In [ ]:
# Load the pretrained giant WITH its 1000-class head
oracle = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1).to(device)
oracle.eval()

# The label names for ImageNet's 1000 classes
imagenet_labels = models.ResNet18_Weights.IMAGENET1K_V1.meta['categories']

# ImageNet preprocessing (must match how the giant was trained)
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # 3 # three chanels (RGB)
])

In [ ]:
from PIL import Image

try:
    photo = Image.open('my_photo.jpg').convert('RGB')
    print("📸 Your photo loaded!")
except FileNotFoundError:
    from sklearn.datasets import load_sample_image
    photo = Image.fromarray(load_sample_image('china.jpg'))
    print("🏯 No photo found — using the backup image.")

plt.imshow(photo); plt.axis('off'); plt.title('The Oracle is looking... 🔮'); plt.show()

x = preprocess(photo).unsqueeze(0).to(device)
with torch.no_grad():
    probs = F.softmax(oracle(x), dim=1)[0]
top5 = torch.topk(probs, 5)

print("\n🔮 The Oracle's top 5 guesses:")
for prob, idx in zip(top5.values, top5.indices):
    print(f"  {imagenet_labels[idx]:25s} {prob*100:5.1f}%")

### 🎯 Mini Exercise 1.1
Try 2–3 different photos. When is the Oracle scary-good? When does it fail? (Try something ambiguous, or a cartoon 😈)

### 📂 Reload the Battle Data: Cats vs Dogs (same as Lab 2) 🐱🐶

In [ ]:
# ResNet needs 224x224, 3-channel, ImageNet-normalized inputs. We resize the tiny CIFAR images up.
tf_resnet = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

trainset_full = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=tf_resnet)
testset_full  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=tf_resnet)

def cats_and_dogs(dataset, n_per_class):
    targets = np.array(dataset.targets)
    cat_idx = np.where(targets == 3)[0][:n_per_class]
    dog_idx = np.where(targets == 5)[0][:n_per_class]
    idx = np.concatenate([cat_idx, dog_idx])
    subset = torch.utils.data.Subset(dataset, idx.tolist())
    return [(img, 0 if lbl == 3 else 1) for img, lbl in subset]

trainset = cats_and_dogs(trainset_full, 2000)
testset  = cats_and_dogs(testset_full, 1000)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)
testloader  = torch.utils.data.DataLoader(testset,  batch_size=64, shuffle=False)
print(f"🐱🐶 Training: {len(trainset)}, Test: {len(testset)} — identical to Lab 2")

### 📋 Enter Your Lab 2 Results
Type in YOUR two accuracy numbers from Lab 2 — the champion must beat YOUR models, not some defaults. 😤

In [ ]:
scratch_acc = 0.66   # TO-DO: your Lab 2 from-scratch test accuracy
aug_acc     = 0.70   # TO-DO: your Lab 2 augmented test accuracy

### 🦾 Act 2: The Transplant — Transfer Learning, Step by Step

1. Load ResNet18 pretrained on ImageNet
2. **Freeze** every layer — the giant's visual skills stay untouched 🔒
3. **Replace the final layer** (`fc`): ImageNet's had 1000 outputs; ours needs just 2 (cat/dog)
4. Train ONLY that new layer

The new `fc` layer is created unfrozen by default, so only it will learn.

**📐 The architecture you're about to build:**

*This is the same diagram from the theory session — now let's code it.*

In [ ]:
transfer_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

for param in transfer_model.parameters():   # freeze the whole giant 🔒
    param.requires_grad = False

# Replace the final fully-connected layer with a fresh 2-class one (unfrozen by default)
transfer_model.fc = nn.Linear(transfer_model.fc.in_features, 2)
transfer_model = transfer_model.to(device)

total = sum(p.numel() for p in transfer_model.parameters())
trainable = sum(p.numel() for p in transfer_model.parameters() if p.requires_grad)
print(f"🔒 Frozen (borrowed) weights: {total - trainable:,}")
print(f"🎓 Weights WE actually train: {trainable:,}  ← just the new head!")

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            preds = model(images).argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

criterion = nn.CrossEntropyLoss()
# Only optimize parameters that require gradients (the new head)
optimizer = optim.Adam([p for p in transfer_model.parameters() if p.requires_grad], lr=0.001)

for epoch in range(3):
    transfer_model.train()
    for images, labels in tqdm(trainloader, desc=f"Epoch {epoch+1}/3", leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(transfer_model(images), labels)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}  |  test accuracy {evaluate(transfer_model, testloader)*100:.1f}%")

transfer_acc = evaluate(transfer_model, testloader)
print(f"\n🦾 Transfer learning test accuracy: {transfer_acc*100:.1f}%")

### 👑 The Scoreboard — Three Models, Same Data

In [ ]:
names = ['From scratch 🥊\n(Lab 2)', 'With augmentation 💪\n(Lab 2)', 'Transfer learning 🦾\n(Lab 3)']
scores = [scratch_acc * 100, aug_acc * 100, transfer_acc * 100]

plt.figure(figsize=(8, 5))
bars = plt.bar(names, scores, color=['indianred', 'goldenrod', 'seagreen'])
for bar, s in zip(bars, scores):
    plt.text(bar.get_x() + bar.get_width()/2, s + 0.5, f'{s:.1f}%', ha='center', fontweight='bold')
plt.ylabel('Test accuracy (%)'); plt.ylim(50, 100)
plt.title('👑 Same data. Same afternoon. Different shoulders.')
plt.axhline(50, color='gray', linestyle='--', label='Coin flip (50%)')
plt.legend(); plt.show()

print(f"🏆 Transfer learning beats your best Lab 2 model by {(transfer_acc - max(scratch_acc, aug_acc))*100:.1f} points!")

### 🎯 Mini Exercise 2.1 — Judge It Yourself 👀
Green titles = correct, red = mistakes. Are the mistakes... understandable? 😅

In [ ]:
# Unnormalize helper so images look natural again
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
classes = ('cat 🐱', 'dog 🐶')

transfer_model.eval()
images = torch.stack([testset[i][0] for i in range(12)])
true   = [testset[i][1] for i in range(12)]
with torch.no_grad():
    preds = transfer_model(images.to(device)).argmax(1).cpu()

fig, axes = plt.subplots(2, 6, figsize=(15, 5))
for ax, img, p, t in zip(axes.flat, images, preds, true):
    shown = (img * std + mean).permute(1, 2, 0).clamp(0, 1)
    ax.imshow(shown); ax.axis('off')
    ax.set_title(classes[p], color='green' if p == t else 'red')
plt.suptitle('Transfer model predictions — green = correct, red = wrong')
plt.show()

### 🎯 Mini Exercise 2.2 — Starve the Models 😈
Rebuild the data with only **100 images per class** and retrain the transfer model. It should barely flinch, while a from-scratch model would collapse. Why does borrowing a giant's eyes make tiny datasets workable?

In [ ]:
# TO-DO: trainset_small = cats_and_dogs(trainset_full, 100), retrain the transfer head, compare


## The GOLDEN Question 🏆

ResNet18 was trained on ImageNet: photos of animals, cars, food, furniture...

**Your friend wants the SAME transfer trick for classifying chest X-rays. 🩻 Will it work as spectacularly as cats vs dogs? Better? Worse? What does your answer reveal about WHEN transfer learning shines?**

*Hint: what visual skills did the giant learn from ImageNet — and how many X-rays were in it?*